# 04. Optimization by gradient descent

In this tutorial, we will introduce optimization of neuron models using gradient descent in Dendra. This allows us to fit model parameters to experimental data or desired behaviors efficiently.

## General principles

Optimization in Dendra is performed using gradient descent algorithms, which iteratively adjust model parameters to minimize a loss function. The loss function quantifies the difference between the model's output and the target data. Common loss functions include mean squared error for fitting voltage traces or spike timing errors for fitting spike trains.

To perform optimization, we need to define:
- A neuron / network model with parameters to optimize.
- A loss function that measures the discrepancy between model output and target data.
- An optimization algorithm (e.g., SGD, Adam) to update the parameters based on the computed gradients.

## Example: Fitting a single neuron model to voltage traces

Here, we will fit a simple neuron model to a target voltage trace using mean squared error as the loss function. For the purposes of this exercise, we will use another model as the "target" to generate synthetic data.

First, we set up the reference model and generate the target voltage trace.

In [ ]:
import dendra as dn
from dendra.models.mod import hh
from dendra.models.callbacks import Recorder
from dendra.units import nA

dn.set_jit_enabled(True)  # True to enable jit compilation globally

ref = dn.Unmyelinated([2.0], celsius=6.3, v_init=-65.0, rhoa=100.0).double()
ref.insert(hh)

rec = Recorder(['v'])

# add injection to generate data
stim_ref = dn.mono_rect(amp=2.0*nA, delay=0.1, pw=0.2)
ref[0, 10].inject(stim_ref)

# generate data
ref.initialize()
ref.run(tstop=5.0, dt=0.005, callbacks=[rec])

v_ref = rec.stack('v')

Now let's set up the model we want to optimize. We will define a simple neuron model with parameters that we can adjust. We will deliberately choose initial parameters that are different from the target model to see how well optimization can recover them.

In [ ]:
bad_model = dn.Unmyelinated([2.0], celsius=6.3, v_init=-65.0, rhoa=100.0).double()
bad_model.insert(hh, gnabar=0.05, gkbar=0.05) # deliberately wrong conductances
bad_rec = Recorder(['v'])
stim_test = dn.mono_rect(amp=2.0*nA, delay=0.1, pw=0.2)
bad_model[0, 10].inject(stim_test)

They key thing we need to do to use this model for optimization is to ensure that the parameters we want to optimize are marked as trainable, and that the model itself is set to training mode. This allows Dendra to compute gradients with respect to these parameters during the optimization process.

To set the model to training mode, we can use the `train()` method provided by Dendra models. This method prepares the model for optimization by enabling gradient tracking for the trainable parameters.

In [ ]:
bad_model.train()

To mark specific parameters as trainable, we can set their `requires_grad` attribute to `True`. This tells Dendra that we want to compute gradients with respect to these parameters during the optimization process. Dendra implements the convenience method `unfreeze()` to do this for us. Before doing this, we need to ensure that the model is built.

In [ ]:
bad_model.build()
bad_model.unfreeze_('hh.gnabar', 'hh.gkbar')
print("\n", bad_model.pretty(show_mechanism_parameters=True))

Now we need to define the loss function that we will use to quantify the difference between the model's output and the target voltage trace. We will use mean squared error (MSE) as our loss function.

In [ ]:
def mse_loss(v_pred, v_target):
    return ((v_pred - v_target) ** 2).mean()

For starters, lets just see if we can run the bad model, calculate the loss, and compute gradients with respect to the parameters we want to optimize.

In [ ]:
bad_model.initialize()
bad_model.run(tstop=5.0, dt=0.005, callbacks=[bad_rec])
v_bad = bad_rec.stack('v')
loss = mse_loss(v_bad, v_ref)
print(f'Initial loss: {loss.item()}')
loss.backward()
print(f'Gradients:')
for name, param in bad_model.named_parameters():
    if param.requires_grad:
        print(f'  {name}: {param.grad}')

Let's also visualize the difference between the bad model's voltage trace and the reference trace before optimization.

In [ ]:
import matplotlib.pyplot as plt

# Recompute after the final optimizer update.
with torch.no_grad():
    bad_model.initialize()
    bad_rec.reset()
    bad_model.run(tstop=5.0, dt=0.005, callbacks=[bad_rec])
    v_bad = bad_rec.stack('v')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# proximal voltage trace
ax1.plot(v_bad.detach().numpy()[:, 0, 0], linewidth=4)
ax1.plot(v_ref.detach().numpy()[:, 0, 0], linewidth=4, linestyle='--')
ax1.set_xlabel('Time step')
ax1.set_ylabel('Voltage (mV)')
ax1.legend(['Bad Model', 'Reference'])
ax1.set_title('Voltage Trace Comparison @ Compartment 0')

ax2.plot(v_bad.detach().numpy()[:, 0, -1], linewidth=4)
ax2.plot(v_ref.detach().numpy()[:, 0, -1], linewidth=4, linestyle='--')
ax2.set_xlabel('Time step')
ax2.set_ylabel('Voltage (mV)')
ax2.legend(['Bad Model', 'Reference'])
ax2.set_title('Voltage Trace Comparison @ Compartment -1')
plt.show()

Ok great, now that we have confirmed that we can compute gradients, we can set up the optimization loop. For this example, we will use the Adam optimizer from PyTorch to update the model parameters based on the computed gradients.

In [ ]:
import torch
optimizer = torch.optim.Adam(bad_model.parameters(), lr=5e-3)

In [ ]:
for step in range(500):
    optimizer.zero_grad()
    bad_model.initialize()
    bad_rec.reset()
    bad_model.run(tstop=5.0, dt=0.005, callbacks=[bad_rec])
    v_bad = bad_rec.stack('v')
    loss = mse_loss(v_bad, v_ref)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f'Step {step}, Loss: {loss.item()}')

Now lets check how the model's voltage trace looks after optimization.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# proximal voltage trace
ax1.plot(v_bad.detach().numpy()[:, 0, 0], linewidth=4)
ax1.plot(v_ref.detach().numpy()[:, 0, 0], linewidth=4, linestyle='--')
ax1.set_xlabel('Time step')
ax1.set_ylabel('Voltage (mV)')
ax1.legend(['Bad Model', 'Reference'])
ax1.set_title('Voltage Trace Comparison @ Compartment 0')

ax2.plot(v_bad.detach().numpy()[:, 0, -1], linewidth=4)
ax2.plot(v_ref.detach().numpy()[:, 0, -1], linewidth=4, linestyle='--')
ax2.set_xlabel('Time step')
ax2.set_ylabel('Voltage (mV)')
ax2.legend(['Bad Model', 'Reference'])
ax2.set_title('Voltage Trace Comparison @ Compartment -1')
plt.show()

In [ ]:
print('Optimized parameters:')
print(f'gkbar: {bad_model.mech.hh.gkbar.item()}')
print(f'gnabar: {bad_model.mech.hh.gnabar.item()}')

print('Original parameters:')
print(f'gkbar: {ref.mech.hh.gkbar.item()}')
print(f'gnabar: {ref.mech.hh.gnabar.item()}')

Well done! You have successfully optimized a neuron model using gradient descent in Dendra. You can see that the optimized parameters are now much closer to the original parameters of the reference model. This demonstrates the power of gradient-based optimization for fitting neuron models to data.

For the explicit-state functional counterpart, including stateless voltage recording and optional checkpointed or compiled one-step execution, see [`examples/functional_gradient_descent.py`](../../examples/functional_gradient_descent.py) and [Functional Populations](../advanced/A8_functional_populations.rst).